In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
from pytorch_tabnet.tab_model import TabNetClassifier
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('Breast Cancer METABRIC.csv')


target_col = 'Tumor Stage'

if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' does not exist!")
stage4_count = (df[target_col] == 4).sum()
df.loc[df[target_col] == 4, target_col] = 3
df = df.dropna(subset=[target_col])

leakage_features = [
    'Overall Survival (Months)',
    'Overall Survival Status',
    'Patient\'s Vital Status',
    'Relapse Free Status (Months)',
    'Relapse Free Status',
    'Nottingham prognostic index',
]
removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)
missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} features with high missing rate have been removed")

exclude_kw = ['stage', 'unnamed', 'patient id', 'patient\'s vital',
              'survival', 'relapse', 'status']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)

X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)

# Label encoding
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

print(f"Feature matrix: {X.shape}")
print(f"Number of classes: {num_classes}")

print(f"\nTarget class mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  Stage {label} → {i}")

# Feature selection
K_FEATURES = 15

selector = SelectKBest(score_func=mutual_info_classif, k=min(K_FEATURES, len(feature_cols)))
selector.fit(X, y_encoded)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (Score: {score:.4f})")

X_selected = X[selected_features].copy()

# Data splitting
class_counts = pd.Series(y_encoded).value_counts()
print("\nSample count by stage:")
for idx, count in class_counts.sort_index().items():
    stage_name = label_encoder.classes_[idx]
    print(f"  Stage {stage_name}: {count} cases")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining set distribution (before SMOTE):")
for idx, count in pd.Series(y_train).value_counts().sort_index().items():
    stage_name = label_encoder.classes_[idx]
    print(f"  Stage {stage_name}: {count} cases")


min_class_samples = pd.Series(y_train).value_counts().min()
k_neighbors = min(5, min_class_samples - 1)

if k_neighbors >= 1:
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
else:
    X_train_bal, y_train_bal = X_train, y_train

print(f"\nBefore SMOTE: {X_train.shape}")
print(f"After SMOTE: {X_train_bal.shape}")
print(f"Added samples: {X_train_bal.shape[0] - X_train.shape[0]}")

print("\nTraining set distribution (after SMOTE):")
for idx, count in pd.Series(y_train_bal).value_counts().sort_index().items():
    stage_name = label_encoder.classes_[idx]
    print(f"  Stage {stage_name}: {count} cases")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# TabNet
tabnet_params = {
    'n_d': 8,
    'n_a': 8,
    'n_steps': 3,
    'gamma': 1.3,
    'lambda_sparse': 1e-3,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': dict(lr=2e-2),
    'scheduler_params': {"step_size": 10, "gamma": 0.9},
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'mask_type': 'sparsemax',
    'seed': 42,
    'verbose': 1
}

tabnet_model = TabNetClassifier(**tabnet_params)

tabnet_model.fit(
    X_train=X_train_scaled,
    y_train=y_train_bal,
    eval_set=[(X_test_scaled, y_test)],
    eval_name=['test'],
    eval_metric=['accuracy'],
    max_epochs=200,
    patience=20,
    batch_size=256,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

# Cross-validation
cv_tabnet_params = tabnet_params.copy()
cv_tabnet_params['verbose'] = 0

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_selected, y_encoded), 1):
    X_cv_train, X_cv_val = X_selected.iloc[train_idx], X_selected.iloc[val_idx]
    y_cv_train, y_cv_val = y_encoded[train_idx], y_encoded[val_idx]

    cv_scaler = StandardScaler()
    X_cv_train_scaled = cv_scaler.fit_transform(X_cv_train)
    X_cv_val_scaled = cv_scaler.transform(X_cv_val)

    cv_model = TabNetClassifier(**cv_tabnet_params)
    cv_model.fit(
        X_train=X_cv_train_scaled,
        y_train=y_cv_train,
        max_epochs=50,
        patience=10,
        batch_size=256,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False,
        eval_set=[(X_cv_val_scaled, y_cv_val)],
        eval_metric=['accuracy']
    )

    y_cv_pred = cv_model.predict(X_cv_val_scaled)
    fold_acc = accuracy_score(y_cv_val, y_cv_pred)
    cv_scores.append(fold_acc)
    print(f"  Fold {fold}: {fold_acc:.4f}")

cv_scores = np.array(cv_scores)
print(f"\naverage: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# Feature importance analysis
feature_importances = tabnet_model.feature_importances_

final_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': feature_importances
}).sort_values('importance', ascending=False)

print(f"\nTabNet importance of these {len(selected_features)} features:")
print(final_importance.to_string(index=False))

# Model evaluation
X_train_original_scaled = scaler.transform(X_train)
y_train_pred = tabnet_model.predict(X_train_original_scaled)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall    = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1        = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)

y_test_pred = tabnet_model.predict(X_test_scaled)

test_accuracy  = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_recall    = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1        = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("=" * 70)
print(" " * 20 + "TabNet Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy-test_accuracy):>15.4f}")
print(f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision-test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall-test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print("=" * 70)

y_test_pred_labels = label_encoder.inverse_transform(y_test_pred)
y_test_true_labels = label_encoder.inverse_transform(y_test)

print(classification_report(y_test_true_labels, y_test_pred_labels,
                            target_names=[f"Stage {c}" for c in sorted(np.unique(y_test_true_labels))],
                            zero_division=0))